# Spin Colour Kernel as a Python library: a magnetic nanodisk

This notebook installs the package from PyPI, writes the Hamiltonian of one state, runs it on the GPU,
plots the thermodynamic curves and draws the out-of-plane spin component $s_z$ of the middle layer of the
disk at several temperatures. It runs as it is on Kaggle, Colab or any machine with an NVIDIA GPU
(compute capability 7.0 or more); on an H100 it takes a few minutes.

In [ ]:
%pip install -q spin-colour-kernel

## 1. The Hamiltonian

The model fixes the kind of spin (`"ising"`, `"xy"` or `"heisenberg"`); every **state** is one set of
parameters of the same Hamiltonian:

$$H = -\sum_s J_s \sum_{\langle ij\rangle_s} \mathbf S_i\cdot\mathbf S_j
      - \sum_{\langle ij\rangle_1,\,xy} \mathbf S_i^{T} A_{ij} \mathbf S_j
      + \sum_i K_{1,i}\,(s_x^2 s_y^2 + s_x^2 s_z^2 + s_y^2 s_z^2) + K_2 \sum_i (s_x s_y s_z)^2
      - \sum_i \mathbf H\cdot\mathbf S_i$$

| Key | Term |
|---|---|
| `J={1: ..., 2: ...}` | exchange per neighbour shell (1 to 4); J > 0 is ferromagnetic |
| `D` | bulk (Bloch) DMI on the in-plane nearest-neighbour bonds |
| `K1`, `Ks`, `K2` | cubic anisotropy in the bulk, on the surface and of second order |
| `H`, `H_theta`, `H_phi` | Zeeman field, its polar angle (0 = +z) and azimuth, in degrees |

A parameter left out is zero (`Ks` defaults to `K1`). `D`, `K1`, `Ks` and `K2` exist for Heisenberg only.
`sck.describe()` lists every setting with its default.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sck

print("spin-colour-kernel", sck.__version__)
print(sck.describe()["hamiltonian"])

## 2. The run

A Heisenberg disk of radius 18.5 cells and 5 layers on a simple cubic lattice, in meV and kelvin, cooled
from 20 K to 0.1 K in steps of 0.1 K, with 10 000 thermalization and 10 000 measurement sweeps per
temperature and 50 replicas. `config_every = 1` keeps the spins of every temperature, which the layer plot
below reads.

In [ ]:
config = dict(
    model="heisenberg", units="meV",
    crystal="sc", box=[39, 39, 5], periodic=[False, False, False],
    coords="cylindrical", shape="rho <= 18.5",
    states=[dict(name="nanodisk", J={1: 1.0}, D=0.88, K1=0.177, Ks=0.139, K2=0.034, H=0.483, H_theta=0.0)],
    T_max=20.0, T_min=0.1, T_step=0.1,
    thermalization=10_000, measurement=10_000, measure_every=10,
    replicas=50, config_every=1, seed=1,
)

report = sck.check(config)
assert report["ok"], report["errors"]
print("sites:", report["sites"])
print("colours:", report["states"][0]["colouring"])
print("temperatures:", report["temperatures"])
print("budget:", report["budget"])

In [ ]:
res = sck.run(config, output_dir="nanodisk_output")
res

## 3. Energy, magnetization, specific heat and susceptibility per site

In [ ]:
fig = res.plot("curves.png")

## 4. The middle layer, $s_z$ only

`sites.npy` holds the position of every active site and `res.configurations()` the saved spins,
shaped (state, replica, saved temperature, site, component). The middle layer is the one at the median
height; each panel shows $s_z$ of one replica, one pixel per site, at one temperature (red up, blue down).

In [ ]:
sites = np.load(res.output_dir / "sites.npy")
spins = res.configurations()                       # memory-mapped, nothing is loaded yet
z = np.round(sites[:, 2], 6)
layers = np.unique(z)
middle = z == layers[len(layers) // 2]
x, y = sites[middle, 0], sites[middle, 1]
print(f"{len(layers)} layers; the middle one (z = {layers[len(layers) // 2]:g}) has {middle.sum()} sites")

# On the simple cubic lattice the sites of a layer sit on a square grid of spacing 1: one pixel per site,
# so the drawing has no gaps at any size. Cells outside the disk stay empty (NaN).
ix = np.round(x - x.min()).astype(int)
iy = np.round(y - y.min()).astype(int)
def layer_image(values):
    img = np.full((iy.max() + 1, ix.max() + 1), np.nan)
    img[iy, ix] = values
    return img
extent = (x.min() - 0.5, x.max() + 0.5, y.min() - 0.5, y.max() + 0.5)

saved_every = int(res.config["config_every"])
T_saved = res.temperatures[::saved_every]
show = [20.0, 18.0, 17.0, 16.0, 15.0, 14.0, 12.0, 10.0, 7.0, 5.0, 2.0, 0.1]
idx = [int(np.argmin(abs(T_saved - t))) for t in show]

replica = 0
fig, axes = plt.subplots(3, 4, figsize=(13, 10), constrained_layout=True)
for ax, k in zip(axes.flat, idx):
    sz = np.asarray(spins[0, replica, k][middle, 2])
    im = ax.imshow(layer_image(sz), origin="lower", extent=extent, cmap="RdBu_r", vmin=-1, vmax=1,
                   interpolation="nearest")
    ax.set_title(f"T = {T_saved[k]:g} K   <s_z> = {sz.mean():+.2f}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.6, label="$s_z$")
fig.suptitle(f"Middle layer, replica {replica}: $s_z$ while cooling")
fig.savefig("middle_layer_sz.png", dpi=150)
plt.show()

The average of $s_z$ over the middle layer and over every replica, at every temperature:

In [ ]:
sz_mean = np.array([np.asarray(spins[0, :, k][:, middle, 2]).mean() for k in range(len(T_saved))])
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(T_saved, sz_mean, "o-", ms=2)
ax.set_xlabel("T (K)"); ax.set_ylabel("<s_z> middle layer"); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("middle_layer_sz_mean.png", dpi=150)
plt.show()